# A tour of the simws 0.8 hub

How one simulation travels through the platform: **job key → isolated adapter → gatekeeper → content-addressed
store → knowledge graph**. Run the cells top to bottom in the `tcad` env (it has DEVSIM):

```bash
cd ~/sim_workspace && PYTHONPATH=. ~/miniforge3/envs/tcad/bin/python -m jupyter lab notebooks/hub_tour.ipynb
```
(or open it in VS Code with that interpreter). Everything is written to a scratch store, so your real store is untouched.

In [ ]:
import tempfile, numpy as np
from simws.hub.jobs import Hub
from simws.hub import adapter as A
hub = Hub(tempfile.mkdtemp(prefix="tour_"))     # scratch store; Hub() alone uses ~/sim_workspace/store
A.names()

## 1. Every adapter declares its contract
The manifest lists outputs with units, the native conventions it converts at its boundary, the validity box,
the physics edges it contributes to the knowledge graph, and citations.

In [ ]:
m = A.manifest("devsim.hbt_gummel")
print(m.description); print("validity:", m.validity)
for o in m.outputs: print(f"  {o.name:12s} [{o.unit}] {o.constraint}")
for e in m.physics_edges: print(f"  edge {e.output} <- {e.input}: {e.mechanism}")

## 2. The job key is the identity of a calculation
Quantities are converted to canonical units before hashing, so `300 K` and `0.3 kK` are the same job.
Defaults that change the answer (pseudopotential files, cut-offs) are pinned into the key at submit time.

In [ ]:
j1 = hub.make_job("materials.intrinsic", {"material": "silicon", "profile": None, "T": ([300.0], "K")})
j2 = hub.make_job("materials.intrinsic", {"material": "silicon", "profile": None, "T": ([0.3], "kK")})
print(j1.id, j1.id == j2.id); j1.key.params

## 3. Submit: run, check, commit — or reuse
A second identical submit is served from the store (`cache: hit`). Results outside the validity box are
kept but flagged `extrapolated`; screening results without an uncertainty are flagged `unverified`.

In [ ]:
r = hub.submit("materials.intrinsic", params={"material": "silicon", "profile": None, "T": ([77.0, 300.0], "K")})
print(r.status, r.flags)
q = r.q("mat.n_i"); print(q.to("cm**-3"), q.flags)
print(hub.submit("materials.intrinsic", params={"material": "silicon", "profile": None, "T": ([77.0, 300.0], "K")}).diagnostics.get("cache"))

## 4. A device: SiGe HBT Gummel sweep (DEVSIM, about 20 s)
Currents arrive in canonical units (A per metre of emitter length) with the passive sign convention.

In [ ]:
g = hub.submit("devsim.hbt_gummel", params={"case": {"ge_peak": 0.15}, "VBE": (np.arange(0.5, 0.81, 0.05), "V"),
                                             "T": (300.0, "K")})
print(g.status, g.flags)
for v, ic, b in zip(g.q("hbt.IC").axes[0].values, g.q("hbt.IC").value, g.q("hbt.beta").value):
    print(f"VBE {v:.2f} V   IC {ic:.3e} A/m   beta {b:6.1f}")

## 5. Failures are categorised, never silent
The gatekeeper and the adapter lifecycle map every outcome to one of: ok, not_converged, timeout,
out_of_memory, invalid_input, outside_validity, engine_error, check_failed.

In [ ]:
bad = hub.submit("materials.intrinsic", params={"material": "unobtainium", "profile": None, "T": ([300.0], "K")})
bad.status, bad.message

## 6. Lineage and staleness in the knowledge graph
Feeding one result into another records a `consumed` edge. If the upstream result is later superseded, everything
downstream reports itself stale.

In [ ]:
a = r
b = hub.submit("materials.intrinsic", params={"material": "silicon", "profile": None, "T": ([350.0], "K")},
               inputs={"upstream": f"{a.id}:mat.Eg"})
print("ancestors of b:", hub.graph.ancestors(f"{b.id}:mat.n_i"))
c = hub.submit("materials.intrinsic", params={"material": "silicon", "profile": None, "T": ([77.0, 301.0], "K")},
               supersedes=a.id)
print("b stale now?", hub.graph.is_stale(f"{b.id}:mat.n_i"))
G = hub.graph.to_networkx(); print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

## 7. Uncertainty: correlated ensembles and budgets
Members share IDs across quantities (`functional=PBE`, `sample=3`), so a ratio of two correlated outputs is
computed member by member. Budgets combine numerical, parametric and model-form parts and name the dominant one.

In [ ]:
from simws.hub import uncertainty as U
from simws.hub.schema import Quantity
Eg = U.Ensemble.from_quantities([Quantity(name="Eg", value=v, unit="eV", member=U.member_id(functional=f))
                                 for f, v in (("LDA", 0.47), ("PBE", 0.58), ("PBEsol", 0.53))])   # illustrative numbers
u = U.budget(1.12, numerical=0.002, parametric=0.004, model_form=float(Eg.range_sigma()))
print(u.sigma, U.dominant(u), u.components)

## 8. Code verification of the device equations
The production Poisson-Boltzmann discretisation, driven by a manufactured solution, must converge at order 2.

In [ ]:
mms = hub.submit("devsim.mms_poisson", params={"levels": 3})
mms.q("mms.observed_order").value, mms.verification["code"]["passed"]

## 9. Bibliography from the results you used

In [ ]:
from simws.hub.citations import bibliography
print(bibliography([r, g, mms]))

## Where next
* `python -m simws.queue add …` / `python -m simws.queue run --until 06:30` — overnight jobs (the nightly task runs the queue).
* `python -m simws.hub.maintain` — prune validated large fields, write a disk report.
* `simws/studies/flagship_a.py` — a full study built only from hub results (rule `flagship_a`).